## Merge_dataset fron 2000 generated scenarios

In [20]:
from pathlib import Path
import pandas as pd
import json

# -------------------- CONFIG --------------------
dataset_dir = Path("scenarios_2000")
output_file = Path("merged_dataset_2000.csv")

# Used to normalize leak severity score
MAX_POSSIBLE_LEAK = 0.01
# ------------------------------------------------

rows = []

scenario_dirs = sorted(dataset_dir.glob("scenario_*"))

print(f"Found {len(scenario_dirs)} scenarios")

if len(scenario_dirs) == 0:
    raise ValueError("No scenario folders found. Check dataset_dir path!")

for scenario_dir in scenario_dirs:

    try:
        scenario_id = scenario_dir.name

        # ---------------- LOAD CSV FILES ----------------
        demands = pd.read_csv(scenario_dir / "demands.csv")
        pressures = pd.read_csv(scenario_dir / "pressures.csv")
        flows = pd.read_csv(scenario_dir / "flows.csv")
        labels = pd.read_csv(scenario_dir / "labels.csv")

        # ---------------- LOAD LEAK TYPE ----------------
        leak_type_file = scenario_dir / "leak_type.json"

        if leak_type_file.exists():
            with open(leak_type_file) as f:
                leak_type = json.load(f).get("leak_type", "unknown")
        else:
            leak_type = "unknown"

        # ---------------- LOAD LEAK INFO ----------------
        leak_info_file = scenario_dir / "leak_info.json"

        leak_severity = 0

        if leak_info_file.exists():
            with open(leak_info_file) as f:
                leak_info = json.load(f)

                if leak_info.get("has_leak"):
                    leak_details = leak_info.get("leak_details", {})
                    leak_severity = leak_details.get("leak_demand_m3s", 0)

        # Normalize leak severity into score
        leak_score = min(leak_severity / MAX_POSSIBLE_LEAK, 1)

        # ---------------- TIMESTEP LOOP ----------------
        for i in range(len(demands)):

            # Demand statistics
            mean_demand = demands.iloc[i].mean()
            max_demand = demands.iloc[i].max()
            min_demand = demands.iloc[i].min()
            std_demand = demands.iloc[i].std()
            demand_range = max_demand - min_demand

            # Pressure statistics
            mean_pressure = pressures.iloc[i].mean()
            max_pressure = pressures.iloc[i].max()
            min_pressure = pressures.iloc[i].min()
            std_pressure = pressures.iloc[i].std()
            pressure_range = max_pressure - min_pressure

            # Flow statistics
            mean_flow = flows.iloc[i].mean()
            max_flow = flows.iloc[i].max()
            min_flow = flows.iloc[i].min()
            std_flow = flows.iloc[i].std()
            flow_range = max_flow - min_flow

            # Hydraulic imbalance indicator
            flow_demand_ratio = mean_flow / (mean_demand + 1e-6)

            # Leak label (1 if any node has leak)
            leak_label = int(labels.iloc[i].max())

            row = {

                # Identifiers
                "scenario": scenario_id,
                "time_index": i,

                # Targets
                "leak_label": leak_label,
                "leak_type": leak_type,
                "leak_score": leak_score,

                # Demand features
                "mean_demand": mean_demand,
                "max_demand": max_demand,
                "min_demand": min_demand,
                "std_demand": std_demand,
                "demand_range": demand_range,

                # Pressure features
                "mean_pressure": mean_pressure,
                "max_pressure": max_pressure,
                "min_pressure": min_pressure,
                "std_pressure": std_pressure,
                "pressure_range": pressure_range,

                # Flow features
                "mean_flow": mean_flow,
                "max_flow": max_flow,
                "min_flow": min_flow,
                "std_flow": std_flow,
                "flow_range": flow_range,

                # Derived hydraulic indicator
                "flow_demand_ratio": flow_demand_ratio
            }

            rows.append(row)

    except Exception as e:
        print(f"⚠ Skipping {scenario_dir.name} due to error: {e}")

# ---------------- CREATE DATAFRAME ----------------

df = pd.DataFrame(rows)

print("\nDataset created")
print("Total rows:", len(df))
print("Total columns:", len(df.columns))

# ---------------- SAVE DATASET ----------------

df.to_csv(output_file, index=False)

print("Dataset saved to:", output_file)

# ---------------- BASIC VALIDATION ----------------

print("\nLeak label distribution:")
print(df["leak_label"].value_counts())

print("\nLeak types:")
print(df["leak_type"].value_counts())

print("\nDataset preview:")
print(df.head())

Found 2000 scenarios

Dataset created
Total rows: 98000
Total columns: 21
Dataset saved to: merged_dataset_2000.csv

Leak label distribution:
45000    2000
86400    2000
46800    2000
48600    2000
50400    2000
52200    2000
54000    2000
55800    2000
57600    2000
59400    2000
61200    2000
63000    2000
64800    2000
66600    2000
68400    2000
70200    2000
72000    2000
73800    2000
75600    2000
77400    2000
79200    2000
81000    2000
82800    2000
1800     2000
43200    2000
41400    2000
19800    2000
3600     2000
5400     2000
7200     2000
9000     2000
10800    2000
12600    2000
14400    2000
16200    2000
18000    2000
21600    2000
39600    2000
23400    2000
25200    2000
27000    2000
28800    2000
30600    2000
32400    2000
34200    2000
36000    2000
37800    2000
84600    2000
0        1953
1          47
Name: leak_label, dtype: int64

Leak types:
continuous      42630
no_leak         27832
pressure        16807
demand           7840
intermittent     2891
Name

 leak_label    Interpretation                                                          
0        No leak occurs in that scenario/timestep (1953 rows) 


1        Leak occurs almost immediately (47 rows)   


1800     Leak occurs **30 minutes** (1800 seconds) into the scenario
                           (2000 rows)
                           
3600     Leak occurs **1 hour** (3600 seconds) into the scenario 
                            (2000 rows)  
                            
5400     Leak occurs **1.5 hours** (5400 seconds) into the scenario                                     (2000 rows)

86400   Leak occurs **24 hours** (86400 seconds) into the scenario                  (2000 rows)  


leak_label column stores the start time of the leak in seconds for each scenario.

For example, if a leak starts at 12:30 PM, that might be 45000 seconds from midnight.

value_counts() counts how many rows have that exact leak start time.

Why 2000?

You have 2000 scenarios where the leak starts at 45000 seconds.

Each scenario contributes 1 row per time step, but when we count the leak start label, each scenario only contributes one value (the leak start time).

So 45000: 2000 means:

Across all scenarios, 2000 of them have their leak starting exactly at 45000 seconds.